# DETECTEPP · Entrenamiento del modelo de EPP (ronda 2)

Entrena un **YOLO11n** con **todas las prendas** en español:
`casco`, `chaleco`, `guantes`, `botas`, `lentes`, `mascarilla` (+ sus `sin_*`), `persona`
y las nuevas de la ronda 2: `careta`, `cofia`, `mandil`, `orejeras`, `respirador`, `traje`.

Datos: tu proyecto **PPE** + **Safevision** (ronda 1) + tu proyecto **EPP ronda 2** (tus fotos).

**Antes de empezar:**
1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** → Guardar.
2. El secreto `ROBOFLOW_API_KEY` ya lo tienes de la ronda 1 (ícono de **llave** a la izquierda, con *Acceso del notebook* activado).

**Cómo usarlo:** ejecuta las celdas **en orden** (▶ o Shift+Enter) y espera a que termine cada una.
Si Colab se desconecta durante el entrenamiento: ejecuta otra vez las celdas 1 a 5 y luego la **6**: continúa donde quedó.

## 1 · Comprobar la GPU
Debe mostrar una tarjeta *Tesla T4*. Si dice que no hay GPU, revisa el paso 1 de arriba.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2 · Instalar librerías
Misma versión de ultralytics que tienes en la laptop (8.4.175), para que el modelo abra sin problemas allá. Tarda 1-2 min.

In [ ]:
!pip install -q ultralytics==8.4.175 roboflow
import ultralytics
ultralytics.checks()

## 3 · Conectar Google Drive
El entrenamiento se guarda en tu Drive (carpeta `detectepp_entrenamiento`) para no perderlo si Colab se desconecta. Acepta los permisos que pida.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/detectepp_entrenamiento')
DRIVE.mkdir(parents=True, exist_ok=True)
print('Guardando en:', DRIVE)

## 4 · Descargar los datasets de Roboflow
Los dos de la ronda 1 y tu proyecto **EPP ronda 2**. Tarda unos minutos.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow
import os

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
os.chdir('/content')
DATASETS = {
    # nombre: (workspace, proyecto, versión)
    'ppe':        ('aaron-isama', 'ppe-vum8g-a26zs', 1),        # tu proyecto: casco, chaleco, guantes, botas
    'safevision': ('andross-workspace', 'safevision-y2mgg', 1), # público: las 6 prendas
    'ronda2':     ('aaron-isama', 'epp-ronda-2', 1),            # tus fotos: careta, cofia, lentes, mandil, orejeras, respirador, traje
}
carpetas = {}
for nombre, (ws, proyecto, version) in DATASETS.items():
    ds = rf.workspace(ws).project(proyecto).version(version).download('yolov8', location=f'/content/datos/{nombre}', overwrite=True)
    carpetas[nombre] = Path(ds.location)
    print(nombre, '->', ds.location)

## 5 · Unir los datasets con nombres en español
- Ronda 1: traduce cada clase a los nombres de DETECTEPP (igual que antes).
- Ronda 2: tus fotos solo tienen marcadas las prendas nuevas. Para que el modelo no "olvide" casco, guantes, botas y los `sin_*`,
  el **modelo de la ronda 1** marca esas prendas en tus fotos (solo si está seguro, ≥ 50 %). Las que sabemos que confundiría
  (tu mandil como chaleco, la cofia como casco, el respirador como mascarilla, tus lentes de uso diario como EPP…) se ignoran.
- Las fotos de entrenamiento de la ronda 2 se usan **2 veces**, porque son menos que las de la ronda 1.

Necesita `epp_yolo.pt` de la ronda 1 en tu Drive (`detectepp_entrenamiento`); si no está, te pedirá subirlo desde
`C:\Users\USER\detector_yolo\models\epp_yolo.pt`.

Al final muestra cuántas etiquetas quedaron por clase: **revisa que ninguna prenda tenga 0**.

In [ ]:
import shutil, yaml
from collections import Counter
from ultralytics import YOLO
from google.colab import files

# Nombre en el dataset (en minúsculas) -> nombre en DETECTEPP. None = se descarta.
TRADUCCION = {
    'hardhat': 'casco', 'helmet': 'casco',
    'vest': 'chaleco', 'gloves': 'guantes', 'boots': 'botas',
    'goggles': 'lentes', 'mask': 'mascarilla',
    'no_hardhat': 'sin_casco', 'no_helmet': 'sin_casco',
    'no_vest': 'sin_chaleco', 'no_gloves': 'sin_guantes', 'no_boots': 'sin_botas',
    'no_goggle': 'sin_lentes', 'no_goggles': 'sin_lentes', 'no_mask': 'sin_mascarilla',
    'person': 'persona',
    'eyeglasses': None, 'sunglasses': None,   # lentes comunes / de sol: NO son EPP
    # ronda 2 (ya vienen en español)
    'careta': 'careta', 'cofia': 'cofia', 'lentes': 'lentes', 'mandil': 'mandil',
    'orejeras': 'orejeras', 'respirador': 'respirador', 'traje': 'traje', 'persona': 'persona',
}
# Las 13 de la ronda 1 conservan su número; las nuevas van al final.
CLASES = ['casco', 'chaleco', 'guantes', 'botas', 'lentes', 'mascarilla',
          'sin_casco', 'sin_chaleco', 'sin_guantes', 'sin_botas', 'sin_lentes', 'sin_mascarilla', 'persona',
          'careta', 'cofia', 'mandil', 'orejeras', 'respirador', 'traje']
ID = {c: i for i, c in enumerate(CLASES)}
DESTINO = Path('/content/datos/epp_unido')
shutil.rmtree(DESTINO, ignore_errors=True)

# --- Modelo de la ronda 1 (para completar las prendas viejas en tus fotos) ---
R1 = DRIVE / 'epp_yolo_r1.pt'          # copia de seguridad: la celda 9 sobrescribe epp_yolo.pt
if not R1.exists():
    if (DRIVE / 'epp_yolo.pt').exists():
        shutil.copy(DRIVE / 'epp_yolo.pt', R1)
    else:
        print('Sube epp_yolo.pt de la ronda 1 (C:\\Users\\USER\\detector_yolo\\models\\epp_yolo.pt):')
        subido = files.upload()
        shutil.copy(next(iter(subido)), R1)
previo = YOLO(str(R1))
print('Modelo ronda 1:', list(previo.names.values()))

# Lo que el modelo de la ronda 1 NUNCA marca en tus fotos (ya está marcado o lo confundiría).
NUNCA = {'persona', 'lentes', 'mascarilla', 'chaleco', 'casco'}   # casco: confunde la cofia con casco
# Por carpeta de origen (inicio del nombre del archivo): lo que sería falso en esas fotos.
POR_CARPETA = {
    'Lentes': {'sin_lentes'},
    'Respirador': {'sin_mascarilla'},
    'Careta': {'sin_casco'},          # la careta va montada en un casco
    'Traje': {'botas', 'sin_botas'},
    'Mandil': {'botas'},              # en esas fotos estás en medias: no son botas
}
CONF_PREVIO = 0.5
REPETIR_R2 = 2

def caja(valores):
    # Polígono (segmentación) -> caja x_centro y_centro ancho alto
    if len(valores) == 4:
        return valores
    xs, ys = valores[0::2], valores[1::2]
    x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
    return [(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1]

def completar_con_r1(img, lineas):
    carpeta = img.name.split('_')[0]
    ignorar = NUNCA | POR_CARPETA.get(carpeta, set())
    r = previo.predict(str(img), imgsz=640, conf=CONF_PREVIO, verbose=False)[0]
    for clase_id, xywhn in zip(r.boxes.cls.tolist(), r.boxes.xywhn.tolist()):
        nombre_r1 = previo.names[int(clase_id)]
        if nombre_r1 in ignorar or nombre_r1 not in ID:
            continue
        lineas.append(' '.join([str(ID[nombre_r1])] + [f'{v:.6f}' for v in xywhn]))
        conteo_r1[nombre_r1] += 1

conteo, conteo_r1, desconocidas = Counter(), Counter(), Counter()
for nombre, carpeta in carpetas.items():
    nombres = yaml.safe_load(open(carpeta / 'data.yaml'))['names']
    if isinstance(nombres, dict):
        nombres = [nombres[k] for k in sorted(nombres)]
    for split_origen, split in (('train', 'train'), ('valid', 'val'), ('test', 'val')):
        imgs = carpeta / split_origen / 'images'
        if not imgs.exists():
            continue
        (DESTINO / 'images' / split).mkdir(parents=True, exist_ok=True)
        (DESTINO / 'labels' / split).mkdir(parents=True, exist_ok=True)
        for img in imgs.iterdir():
            lbl = carpeta / split_origen / 'labels' / (img.stem + '.txt')
            lineas = []
            if lbl.exists():
                for linea in lbl.read_text().split('\n'):
                    p = linea.split()
                    if len(p) < 5 or int(p[0]) >= len(nombres):
                        continue
                    original = nombres[int(p[0])]
                    clave = original.lower().strip()
                    if clave not in TRADUCCION:
                        desconocidas[original] += 1
                        continue
                    nueva = TRADUCCION[clave]
                    if nueva is None:
                        continue
                    lineas.append(' '.join([str(ID[nueva])] + [f'{v:.6f}' for v in caja([float(x) for x in p[1:]])]))
                    conteo[nueva] += 1
            if nombre == 'ronda2':
                completar_con_r1(img, lineas)
            copias = REPETIR_R2 if (nombre == 'ronda2' and split == 'train') else 1
            for n in range(copias):
                nuevo = f'{nombre}_{n}_{img.name}' if copias > 1 else f'{nombre}_{img.name}'
                shutil.copy(img, DESTINO / 'images' / split / nuevo)
                (DESTINO / 'labels' / split / (Path(nuevo).stem + '.txt')).write_text('\n'.join(lineas))
        print(f'{nombre} / {split_origen}: listo')

yaml.safe_dump({'path': str(DESTINO), 'train': 'images/train', 'val': 'images/val', 'names': dict(enumerate(CLASES))},
               open(DESTINO / 'data.yaml', 'w'), allow_unicode=True)
for s in ('train', 'val'):
    print(s, len(list((DESTINO / 'images' / s).iterdir())), 'imágenes')
print('\nEtiquetas por clase (dataset + completadas por la ronda 1):')
for c in CLASES:
    extra = f'  (+{conteo_r1[c]} de la ronda 1 en tus fotos)' if conteo_r1[c] else ''
    print(f'  {c:15s} {conteo[c]}{extra}')
if desconocidas:
    print('\n⚠ Clases sin traducción (se descartaron):', dict(desconocidas))

## 6 · Entrenar
⏱ 1,5 a 2 horas. Puedes dejar la pestaña abierta y hacer otra cosa (no cierres el navegador).
Si se desconecta: ejecuta otra vez las celdas 1 a 5 y luego esta: **retoma desde el último ciclo guardado**.

Parte del modelo de la ronda 1 (ya sabe casco, guantes, etc.) y aprende las prendas nuevas.
Qué mirar mientras corre: la columna **mAP50** debe ir subiendo.

In [ ]:
RUNS = DRIVE / 'runs'
ultimo = RUNS / 'epp_r2' / 'weights' / 'last.pt'

if ultimo.exists():
    print('Continuando entrenamiento anterior:', ultimo)
    YOLO(str(ultimo)).train(resume=True)
else:
    YOLO(str(R1)).train(                      # parte de la ronda 1; la cabeza se amplía a 19 clases
        data=str(DESTINO / 'data.yaml'),
        epochs=80, imgsz=640, batch=32, patience=20,
        project=str(RUNS), name='epp_r2', exist_ok=True,
        workers=2, plots=True,
    )

## 7 · Ver resultados por prenda
`mAP50` por clase: más cerca de 1 = mejor. Si alguna prenda sale muy baja (menos de 0,4), avísame.

In [ ]:
mejor = RUNS / 'epp_r2' / 'weights' / 'best.pt'
modelo = YOLO(str(mejor))
print('Clases del modelo:', modelo.names)
m = modelo.val(data=str(DESTINO / 'data.yaml'), imgsz=640, plots=False)
print(f'\nmAP50 general: {m.box.map50:.3f}')
for i, c in modelo.names.items():
    if i in m.box.ap_class_index:
        k = list(m.box.ap_class_index).index(i)
        print(f'  {c:15s} {m.box.ap50[k]:.3f}')

## 8 · Probar con una foto tuya (opcional)
Sube una foto con EPP y mira las cajas.

In [ ]:
from google.colab import files
from IPython.display import Image, display
subidas = files.upload()
for nombre in subidas:
    r = modelo.predict(nombre, imgsz=640, conf=0.45)[0]
    r.save(filename='prediccion.jpg')
    display(Image('prediccion.jpg', width=640))

## 9 · Descargar el modelo
Exporta a **ONNX** (el formato que usa DETECTEPP: corre sin torch y no congela la ventana) y descarga
`epp_yolo.onnx` y `epp_yolo.pt` (también quedan copiados en tu Drive, en `detectepp_entrenamiento`).

En la laptop: copia los dos archivos a `C:\Users\USER\detector_yolo\models\` y luego dime para revisarlo.

In [ ]:
onnx = Path(YOLO(str(mejor)).export(format='onnx', imgsz=640, opset=12, simplify=True))
for origen, nombre in ((mejor, 'epp_yolo.pt'), (onnx, 'epp_yolo.onnx')):
    shutil.copy(origen, DRIVE / nombre)
    shutil.copy(origen, f'/content/{nombre}')
    files.download(f'/content/{nombre}')
print('También en Drive:', DRIVE)